# RoBERTa-large vs. TAPT RoBERTa-large — seed 13

This notebook runs a controlled downstream comparison between the original `FacebookAI/roberta-large` encoder and the seed-13 task-adapted checkpoint. Both models use the same seed, split, first-token pooling, classifier hyperparameters, loss, scheduler, and early-stopping rule. The only intended difference is the starting encoder checkpoint.

Results are written to `Code/BERT_models/experiments/tapt/downstream_comparison`. Existing results from the earlier random-pooler architecture are not reused.

In [ ]:
from pathlib import Path
import json

PROJECT_DIR = next(
    (root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / 'data' / 'annotate-lime.csv').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not locate the project root containing data/annotate-lime.csv.')
SOURCE_NOTEBOOK = PROJECT_DIR / 'Code' / 'BERT_models' / 'notebooks' / 'training' / 'BERT_Soft_Label_LR_Epoch_Tuning.ipynb'

source_notebook = json.loads(SOURCE_NOTEBOOK.read_text())
loaded_cells = 0
for cell_index, cell in enumerate(source_notebook['cells'], start=1):
    if cell.get('cell_type') != 'code':
        continue
    source = ''.join(cell.get('source', []))
    if source.lstrip().startswith('SMOKE_TEST = False'):
        break
    exec(compile(source, f'{SOURCE_NOTEBOOK.name}:cell-{cell_index}', 'exec'), globals())
    loaded_cells += 1

print(f'Loaded {loaded_cells} setup/definition cells from {SOURCE_NOTEBOOK.name}.')

In [ ]:
COMPARISON_SEED = 13
COMPARISON_OUTPUT_DIR = PROJECT_DIR / 'Code' / 'BERT_models' / 'experiments' / 'tapt' / 'downstream_comparison'
TAPT_CHECKPOINT_DIR = (
    PROJECT_DIR
    / 'Code'
    / 'BERT_models'
    / 'experiments'
    / 'tapt'
    / 'masked_language_modeling'
    / 'roberta_large_tapt_random_seed13'
)

baseline_config = with_tuning_settings(
    ROBERTA_LARGE_CONFIG,
    learning_rate=1e-5,
    num_epochs=30,
    patience=6,
    scheduler_type='linear',
)

tapt_config = with_tuning_settings(
    make_roberta_large_tapt_config(TAPT_CHECKPOINT_DIR),
    learning_rate=1e-5,
    num_epochs=30,
    patience=6,
    scheduler_type='linear',
)

comparison_configs = [baseline_config, tapt_config]
COMPARISON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fail before loading either large model if the TAPT checkpoint is incomplete or mismatched.
comparison_train_df, _, _ = make_random_split(dataset, COMPARISON_SEED, split_scheme='majority')
validate_tapt_checkpoint_for_run(tapt_config, f'random_seed{COMPARISON_SEED}', comparison_train_df)

display(pd.DataFrame([asdict(config) for config in comparison_configs]))
print('Comparison output directory:', COMPARISON_OUTPUT_DIR)
print('Validated TAPT checkpoint:', TAPT_CHECKPOINT_DIR)

In [ ]:
# Safe defaults: Run All reads completed results without retraining classifiers.
RUN_TAPT_COMPARISON = False
SKIP_COMPLETED_COMPARISON = True
COMPARISON_SUMMARY_PATH = COMPARISON_OUTPUT_DIR / 'roberta_large_tapt_comparison_seed13.csv'

if RUN_TAPT_COMPARISON:
    comparison_summary = run_random_split_suite(
        model_configs=comparison_configs,
        seeds=[COMPARISON_SEED],
        loss_mode='binomial_nll',
        output_dir=COMPARISON_OUTPUT_DIR,
        summary_name='roberta_large_tapt_comparison_seed13',
        skip_completed=SKIP_COMPLETED_COMPARISON,
    )
else:
    comparison_summary = (
        pd.read_csv(COMPARISON_SUMMARY_PATH)
        if COMPARISON_SUMMARY_PATH.exists()
        else pd.DataFrame()
    )
    print('Comparison training is disabled. Set RUN_TAPT_COMPARISON = True to opt in.')
    if not comparison_summary.empty:
        print('Loaded existing comparison:', COMPARISON_SUMMARY_PATH)

comparison_columns = [
    'experiment_name',
    'model_name',
    'run_id',
    'pooling',
    'best_epoch',
    'best_val_loss',
    'macro_mae',
    'mean_pearson',
    'majority_tuned_macro_f1',
]
if comparison_summary.empty:
    comparison_table = pd.DataFrame(columns=comparison_columns)
    print('No completed comparison results were found.')
else:
    comparison_table = comparison_summary[comparison_columns].sort_values('experiment_name')
    display(comparison_table)
    if RUN_TAPT_COMPARISON:
        comparison_table.to_csv(
            COMPARISON_OUTPUT_DIR / 'roberta_large_tapt_comparison_seed13_key_metrics.csv',
            index=False,
        )

## Interpretation

For the primary soft-label comparison, lower `best_val_loss` and `macro_mae` are better, while higher `mean_pearson` is better. `majority_tuned_macro_f1` is a secondary thresholded metric. Because this is one seed, treat it as a controlled seed-13 comparison rather than a final estimate of variability.

## Final results summary

This controlled seed-13 experiment compared standard `FacebookAI/roberta-large` with the seed-13 TAPT encoder. Both models used the same 424/91/91 train-validation-test split, first-token pooling, batch size 4, learning rate `1e-5`, linear scheduler, maximum of 30 epochs, and early-stopping patience of 6. The only intended difference was whether the encoder had first undergone task-adaptive masked-language-model training.

Standard RoBERTa reached its best validation loss at epoch 6 (`0.4008`) and stopped at epoch 12. TAPT RoBERTa reached its best validation loss at epoch 8 (`0.3979`) and stopped at epoch 14. On the test set, standard versus TAPT results were: macro MAE `0.1332` versus `0.1338`, macro RMSE `0.1891` versus `0.1883`, mean Pearson `0.6025` versus `0.5893`, mean Spearman `0.5522` versus `0.5851`, and majority tuned macro-F1 `0.4530` versus `0.4545`.

The differences are small and point in different directions. TAPT fit the validation data slightly better and ranked examples somewhat better according to Spearman correlation, but standard RoBERTa had slightly lower absolute test error and higher Pearson correlation. Majority macro-F1 was effectively tied. Therefore, this single split provides no clear evidence that TAPT improves the downstream task. Multiple matched seeds are required before making a general claim; leave-one-situation-out evaluation would be needed to assess transfer to completely unseen situations.

For a results-only explanation with plots and metric definitions, open `RoBERTa_TAPT_Results_Comparison_Seed13.ipynb`.